# NSE Daily Stocks + NIFTY 50 Data Sync

Idempotent Google Colab notebook for:

- NSE Cash Market daily stock data
- NIFTY 50 daily index OHLC data
- Missing-date synchronization
- Existing-file validation
- Corrupt-file repair
- Parquet output
- CSV manifests
- DuckDB sanity checks

## Storage

```text
/content/drive/MyDrive/quant/data/
├── parquet/                  # NSE stock data
│   └── year=YYYY/
│       └── nse_cm_YYYYMMDD.parquet
└── indices/
    └── nifty50/              # NIFTY 50 index data
        └── year=YYYY/
            └── nifty50_YYYYMMDD.parquet
```

The stock dataset keeps the existing schema:

`date, symbol, open, high, low, close, volume`

NIFTY 50 uses the same schema. Since an index itself is not a traded security, its `volume` is stored as null.


In [ ]:
# ============================================================
# 1. SETUP
# ============================================================

!pip -q install pandas pyarrow requests duckdb tqdm

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from datetime import date, datetime, timedelta
import io
import time
import zipfile
import logging

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm.auto import tqdm
import duckdb

print("Environment ready.")


In [ ]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

BASE_DIR = Path("/content/drive/MyDrive/quant")
DATA_DIR = BASE_DIR / "data"

# Existing stock data
PARQUET_DIR = DATA_DIR / "parquet"
RAW_DIR = DATA_DIR / "raw" / "nse_bhavcopy"
METADATA_DIR = DATA_DIR / "metadata"
MANIFEST_FILE = DATA_DIR / "download_manifest.csv"

# NIFTY 50 data
INDEX_DATA_DIR = DATA_DIR / "indices"
NIFTY50_DIR = INDEX_DATA_DIR / "nifty50"
NIFTY50_MANIFEST_FILE = INDEX_DATA_DIR / "nifty50_manifest.csv"

# Date range
START_DATE = date(2011, 1, 1)
END_DATE = None   # None = today

# NSE stock format transition
LEGACY_END_DATE = date(2024, 7, 5)
UDIFF_START_DATE = date(2024, 7, 8)

# Optional stock filter
# None = all stocks
# Example: {"RELIANCE", "TCS", "INFY"}
STOCKS = None

# Download behavior
REQUEST_TIMEOUT = 60
MAX_RETRIES = 5
BACKOFF_FACTOR = 1.5
SLEEP_BETWEEN_REQUESTS = 0.10

# Repair invalid/corrupt Parquet files
REPAIR_INVALID_FILES = True

# Keep downloaded raw ZIPs?
KEEP_RAW_ZIPS = False

EXPECTED_COLUMNS = [
    "date",
    "symbol",
    "open",
    "high",
    "low",
    "close",
    "volume",
]

# Create directories
for directory in [
    BASE_DIR,
    DATA_DIR,
    PARQUET_DIR,
    RAW_DIR,
    METADATA_DIR,
    INDEX_DATA_DIR,
    NIFTY50_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

if END_DATE is None:
    END_DATE = date.today()

assert START_DATE <= END_DATE

print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)
print("STOCK DATA:", PARQUET_DIR)
print("NIFTY50 DATA:", NIFTY50_DIR)
print("DATE RANGE:", START_DATE, "to", END_DATE)


In [ ]:
# ============================================================
# 3. LOGGING + HTTP SESSION
# ============================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

logger = logging.getLogger("nse_sync")

session = requests.Session()

retry = Retry(
    total=MAX_RETRIES,
    connect=MAX_RETRIES,
    read=MAX_RETRIES,
    status=MAX_RETRIES,
    backoff_factor=BACKOFF_FACTOR,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"]),
    raise_on_status=False,
)

adapter = HTTPAdapter(max_retries=retry)

session.mount("https://", adapter)
session.mount("http://", adapter)

session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0 Safari/537.36"
    ),
    "Accept": "*/*",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.nseindia.com/",
    "Connection": "keep-alive",
})

print("HTTP session configured.")


In [ ]:
# ============================================================
# 4. COMMON DATE / PATH HELPERS
# ============================================================

def trading_day_candidates(start_date: date, end_date: date):
    current = start_date

    while current <= end_date:
        if current.weekday() < 5:
            yield current

        current += timedelta(days=1)


def parquet_path_for_day(day: date) -> Path:
    year_dir = PARQUET_DIR / f"year={day.year}"
    year_dir.mkdir(parents=True, exist_ok=True)

    return year_dir / f"nse_cm_{day:%Y%m%d}.parquet"


def nifty50_parquet_path_for_day(day: date) -> Path:
    year_dir = NIFTY50_DIR / f"year={day.year}"
    year_dir.mkdir(parents=True, exist_ok=True)

    return year_dir / f"nifty50_{day:%Y%m%d}.parquet"


def raw_zip_path_for_day(day: date) -> Path:
    return RAW_DIR / f"nse_cm_{day:%Y%m%d}.zip"


print("Path helpers ready.")


In [ ]:
# ============================================================
# 5. NSE STOCK URL HELPERS
# ============================================================

def legacy_url(day: date) -> str:
    month = day.strftime("%b").upper()

    filename = (
        f"cm{day:%d}{month}{day:%Y}bhav.csv.zip"
    )

    return (
        "https://nsearchives.nseindia.com/content/"
        f"historical/EQUITIES/{day.year}/{month}/{filename}"
    )


def udiff_url(day: date) -> str:
    filename = (
        f"BhavCopy_NSE_CM_0_0_0_"
        f"{day:%Y%m%d}_F_0000.csv.zip"
    )

    return (
        "https://nsearchives.nseindia.com/content/cm/"
        f"{filename}"
    )


def url_for_day(day: date) -> str:
    if day <= LEGACY_END_DATE:
        return legacy_url(day)

    return udiff_url(day)


for d in [
    date(2011, 1, 3),
    date(2024, 7, 5),
    date(2024, 7, 8),
]:
    print(d, "->", url_for_day(d))


In [ ]:
# ============================================================
# 6. STOCK DATA NORMALIZATION
# ============================================================

def normalize_column_name(name: str) -> str:
    return (
        str(name)
        .strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
        .replace(".", "_")
        .replace("/", "_")
    )


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename = {}

    for col in df.columns:
        n = normalize_column_name(col)

        aliases = {
            "tradingsymbol": "symbol",
            "symbol": "symbol",

            "timestamp": "date",
            "trade_date": "date",
            "date": "date",

            "open_price": "open",
            "open": "open",

            "high_price": "high",
            "high": "high",

            "low_price": "low",
            "low": "low",

            "close_price": "close",
            "close": "close",

            "last_price": "close",
            "ltp": "close",

            "tottrdqty": "volume",
            "total_traded_quantity": "volume",
            "total_traded_qty": "volume",
            "volume": "volume",
        }

        if n in aliases:
            rename[col] = aliases[n]

    df = df.rename(columns=rename)

    missing = [
        c for c in EXPECTED_COLUMNS
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Missing required columns: {missing}. "
            f"Received columns: {list(df.columns)}"
        )

    df = df[EXPECTED_COLUMNS].copy()

    df["date"] = pd.to_datetime(
        df["date"],
        errors="coerce",
    ).dt.date

    df["symbol"] = (
        df["symbol"]
        .astype("string")
        .str.strip()
    )

    for col in [
        "open",
        "high",
        "low",
        "close",
        "volume",
    ]:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce",
        )

    df = df.dropna(
        subset=["date", "symbol"]
    )

    if STOCKS is not None:
        df = df[
            df["symbol"].isin(STOCKS)
        ]

    df = df.drop_duplicates(
        subset=["date", "symbol"],
        keep="last",
    )

    return df


def read_nse_zip(content: bytes) -> pd.DataFrame:
    with zipfile.ZipFile(
        io.BytesIO(content)
    ) as z:

        csv_names = [
            name
            for name in z.namelist()
            if name.lower().endswith(".csv")
        ]

        if not csv_names:
            raise ValueError(
                "ZIP archive contains no CSV."
            )

        with z.open(csv_names[0]) as f:
            df = pd.read_csv(f)

    return normalize_columns(df)


In [ ]:
# ============================================================
# 7. STOCK VALIDATION + ATOMIC PARQUET WRITE
# ============================================================

def validate_dataframe(
    df: pd.DataFrame,
    expected_day: date,
) -> tuple[bool, str]:

    if df.empty:
        return False, "empty_dataframe"

    missing = [
        c for c in EXPECTED_COLUMNS
        if c not in df.columns
    ]

    if missing:
        return False, f"missing_columns:{missing}"

    if df["date"].isna().any():
        return False, "null_dates"

    if not (
        df["date"] == expected_day
    ).all():
        return False, "wrong_date"

    if (
        df["symbol"].isna().any()
        or (
            df["symbol"]
            .astype(str)
            .str.len()
            == 0
        ).any()
    ):
        return False, "invalid_symbols"

    numeric_cols = [
        "open",
        "high",
        "low",
        "close",
        "volume",
    ]

    if df[numeric_cols].isna().all(axis=1).any():
        return False, "rows_with_all_numeric_values_null"

    return True, "ok"


def validate_parquet(
    path: Path,
    expected_day: date,
) -> tuple[bool, str, int]:

    if not path.exists():
        return False, "missing_file", 0

    try:
        table = pq.read_table(path)
        df = table.to_pandas()

        df = normalize_columns(df)

        ok, reason = validate_dataframe(
            df,
            expected_day,
        )

        return ok, reason, len(df)

    except Exception as exc:

        return (
            False,
            f"parquet_read_error:{type(exc).__name__}:{exc}",
            0,
        )


def atomic_write_parquet(
    df: pd.DataFrame,
    path: Path,
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    tmp_path = path.with_suffix(
        ".parquet.tmp"
    )

    tmp_path.unlink(
        missing_ok=True
    )

    table = pa.Table.from_pandas(
        df,
        preserve_index=False,
    )

    pq.write_table(
        table,
        tmp_path,
        compression="snappy",
    )

    ok, reason, rows = validate_parquet(
        tmp_path,
        df["date"].iloc[0],
    )

    if not ok:
        tmp_path.unlink(
            missing_ok=True
        )

        raise ValueError(
            f"Post-write validation failed: {reason}"
        )

    tmp_path.replace(path)

    return rows


In [ ]:
# ============================================================
# 8. STOCK DOWNLOAD
# ============================================================

def download_stock_day(day: date) -> dict:

    path = parquet_path_for_day(day)
    url = url_for_day(day)

    if path.exists():

        ok, reason, rows = validate_parquet(
            path,
            day,
        )

        if ok:
            return {
                "date": day.isoformat(),
                "status": "already_valid",
                "rows": rows,
                "path": str(path),
                "url": url,
                "message": "Existing Parquet passed validation.",
            }

        if not REPAIR_INVALID_FILES:
            return {
                "date": day.isoformat(),
                "status": "invalid_skipped",
                "rows": rows,
                "path": str(path),
                "url": url,
                "message": reason,
            }

        logger.warning(
            "%s invalid: %s — repairing.",
            day,
            reason,
        )

        path.unlink(
            missing_ok=True
        )

    try:

        response = session.get(
            url,
            timeout=REQUEST_TIMEOUT,
        )

        if response.status_code == 404:
            return {
                "date": day.isoformat(),
                "status": "not_found",
                "rows": 0,
                "path": str(path),
                "url": url,
                "message": (
                    "NSE archive not found; "
                    "likely holiday/non-trading day."
                ),
            }

        response.raise_for_status()

        content = response.content

        if not content:
            raise ValueError(
                "NSE returned an empty response."
            )

        if KEEP_RAW_ZIPS:
            raw_zip_path_for_day(
                day
            ).write_bytes(content)

        df = read_nse_zip(content)

        ok, reason = validate_dataframe(
            df,
            day,
        )

        if not ok:
            raise ValueError(
                f"Downloaded data failed validation: {reason}"
            )

        rows = atomic_write_parquet(
            df,
            path,
        )

        return {
            "date": day.isoformat(),
            "status": "downloaded",
            "rows": rows,
            "path": str(path),
            "url": url,
            "message": "Downloaded successfully.",
        }

    except Exception as exc:

        return {
            "date": day.isoformat(),
            "status": "error",
            "rows": 0,
            "path": str(path),
            "url": url,
            "message": (
                f"{type(exc).__name__}: {exc}"
            ),
        }


In [ ]:
# ============================================================
# 9. STOCK MANIFEST
# ============================================================

MANIFEST_COLUMNS = [
    "date",
    "status",
    "rows",
    "path",
    "url",
    "message",
    "checked_at",
]


def load_manifest() -> pd.DataFrame:

    if not MANIFEST_FILE.exists():
        return pd.DataFrame(
            columns=MANIFEST_COLUMNS
        )

    try:

        manifest = pd.read_csv(
            MANIFEST_FILE
        )

        for col in MANIFEST_COLUMNS:
            if col not in manifest.columns:
                manifest[col] = None

        return manifest[
            MANIFEST_COLUMNS
        ]

    except Exception as exc:

        logger.warning(
            "Could not read manifest: %s",
            exc,
        )

        return pd.DataFrame(
            columns=MANIFEST_COLUMNS
        )


manifest = load_manifest()


def append_manifest(record: dict):

    global manifest

    row = {
        col: record.get(col)
        for col in MANIFEST_COLUMNS
    }

    row["checked_at"] = datetime.now().isoformat(
        timespec="seconds"
    )

    manifest = pd.concat(
        [
            manifest,
            pd.DataFrame([row]),
        ],
        ignore_index=True,
    )

    manifest["date"] = (
        manifest["date"].astype(str)
    )

    manifest = (
        manifest
        .drop_duplicates(
            subset=["date"],
            keep="last",
        )
        .sort_values("date")
    )

    manifest.to_csv(
        MANIFEST_FILE,
        index=False,
    )


print(
    "Existing stock manifest records:",
    len(manifest),
)


In [ ]:
# ============================================================
# 10. STOCK SYNC
# ============================================================

candidates = list(
    trading_day_candidates(
        START_DATE,
        END_DATE,
    )
)

existing_valid = 0
existing_invalid = 0
missing = 0

for day in candidates:

    path = parquet_path_for_day(day)

    if not path.exists():
        missing += 1
        continue

    ok, _, _ = validate_parquet(
        path,
        day,
    )

    if ok:
        existing_valid += 1
    else:
        existing_invalid += 1


print("Weekday candidates :", f"{len(candidates):,}")
print("Valid existing     :", f"{existing_valid:,}")
print("Invalid existing   :", f"{existing_invalid:,}")
print("Missing            :", f"{missing:,}")


In [ ]:
# ============================================================
# 11. STOCK DOWNLOAD LOOP — MISSING + FAILED ONLY
# ============================================================

run_started = datetime.now()
run_results = []

# ------------------------------------------------------------
# BUILD DOWNLOAD QUEUE FROM FILESYSTEM
# ------------------------------------------------------------
#
# candidates = ALL dates in the requested historical range.
#
# We DO NOT iterate over candidates directly.
#
# Existing Parquet:
#     -> SKIP permanently
#
# Missing Parquet:
#     -> DOWNLOAD / RETRY
#
# This means a stale manifest can NEVER cause an existing file
# to be downloaded again.
# ------------------------------------------------------------

stock_existing = []
stock_missing = []

for day in candidates:

    path = parquet_path_for_day(day)

    if path.exists():
        stock_existing.append(day)
    else:
        stock_missing.append(day)


# ------------------------------------------------------------
# HARD ASSERTION
# ------------------------------------------------------------

existing_in_missing = [
    day
    for day in stock_missing
    if parquet_path_for_day(day).exists()
]

if existing_in_missing:
    raise RuntimeError(
        "ABORT: dates classified as missing now have Parquet files:\n"
        + "\n".join(
            str(parquet_path_for_day(day))
            for day in existing_in_missing[:100]
        )
    )


# ------------------------------------------------------------
# FAILED / ERROR RETRIES
# ------------------------------------------------------------
#
# A failed manifest entry is useful for classification, but the
# filesystem remains authoritative.
#
# If the file is missing, it belongs in the queue regardless of
# whether the previous manifest says:
#
#     downloaded
#     failed
#     error
#     not_found
#     anything else
#
# This repairs stale/inconsistent manifests automatically.
# ------------------------------------------------------------

download_queue = list(stock_missing)


# ------------------------------------------------------------
# FINAL QUEUE ASSERTION
# ------------------------------------------------------------

queue_existing = [
    day
    for day in download_queue
    if parquet_path_for_day(day).exists()
]

if queue_existing:
    raise RuntimeError(
        "ABORT: download queue contains existing Parquet files:\n"
        + "\n".join(
            str(parquet_path_for_day(day))
            for day in queue_existing[:100]
        )
    )

assert len(stock_existing) + len(stock_missing) == len(candidates)
assert len(download_queue) == len(stock_missing)


# ------------------------------------------------------------
# PREFLIGHT SUMMARY
# ------------------------------------------------------------

print("=" * 90)
print("STOCK DOWNLOAD PREFLIGHT")
print("=" * 90)

print(f"Candidate dates      : {len(candidates):,}")
print(f"Existing Parquets    : {len(stock_existing):,}")
print(f"Missing Parquets     : {len(stock_missing):,}")
print(f"Download queue       : {len(download_queue):,}")

print()

if not download_queue:

    print("✓ NOTHING TO DOWNLOAD")
    print("✓ All candidate dates already have Parquet files.")

else:

    print("✓ Preflight passed.")
    print("✓ Existing files will NOT be downloaded.")
    print("✓ Only missing files will be requested.")
    print()


# ------------------------------------------------------------
# DOWNLOAD ONLY MISSING FILES
# ------------------------------------------------------------

for i, day in enumerate(
    tqdm(
        download_queue,
        desc="NSE missing/failed files",
    ),
    1,
):

    # --------------------------------------------------------
    # FINAL SAFETY CHECK BEFORE NETWORK REQUEST
    # --------------------------------------------------------

    path = parquet_path_for_day(day)

    if path.exists():
        raise RuntimeError(
            f"ABORT: Parquet appeared before network request:\n{path}"
        )


    # --------------------------------------------------------
    # DOWNLOAD
    # --------------------------------------------------------

    result = download_stock_day(day)

    run_results.append(result)

    append_manifest(result)


    # --------------------------------------------------------
    # RATE LIMITING
    # --------------------------------------------------------

    if result["status"] in {
        "downloaded",
        "error",
        "failed",
    }:

        if SLEEP_BETWEEN_REQUESTS:
            time.sleep(
                SLEEP_BETWEEN_REQUESTS
            )


run_finished = datetime.now()

run_df = pd.DataFrame(
    run_results
)


# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print()
print("=" * 90)
print("STOCK SYNC COMPLETED")
print("=" * 90)

print("Started :", run_started)
print("Finished:", run_finished)
print("Elapsed :", run_finished - run_started)

print()
print(f"Existing / skipped : {len(stock_existing):,}")
print(f"Downloaded / retried: {len(run_results):,}")

In [ ]:
# ============================================================
# 12. STOCK SYNC SUMMARY
# ============================================================

if run_df.empty:

    print("No stock dates processed.")

else:

    print("Stock status counts:")

    display(
        run_df["status"]
        .value_counts()
        .rename_axis("status")
        .reset_index(
            name="count"
        )
    )

    downloaded_rows = run_df.loc[
        run_df["status"] == "downloaded",
        "rows",
    ].sum()

    print(
        "Rows downloaded:",
        f"{downloaded_rows:,}",
    )

    errors = run_df[
        run_df["status"] == "error"
    ]

    if not errors.empty:

        print("Stock errors:")

        display(
            errors[
                [
                    "date",
                    "message",
                    "url",
                ]
            ]
        )


In [ ]:
# ============================================================
# 13. NSE SESSION INITIALIZATION FOR NIFTY 50
# ============================================================

def initialize_nse_session():

    try:

        response = session.get(
            "https://www.nseindia.com/",
            timeout=REQUEST_TIMEOUT,
        )

        logger.info(
            "NSE session initialized: HTTP %s",
            response.status_code,
        )

    except Exception as exc:

        logger.warning(
            "NSE session initialization failed: %s",
            exc,
        )


initialize_nse_session()


In [ ]:
# ============================================================
# 14. NIFTY 50 API + NORMALIZATION
# ============================================================

def nifty50_api_url(
    day_from: date,
    day_to: date,
) -> str:

    from_str = day_from.strftime(
        "%d-%m-%Y"
    )

    to_str = day_to.strftime(
        "%d-%m-%Y"
    )

    return (
        "https://www.nseindia.com/api/"
        "historical/indicesHistory"
        "?indexType=NIFTY%2050"
        f"&from={from_str}"
        f"&to={to_str}"
    )


def normalize_nifty50_response(
    payload: dict,
) -> pd.DataFrame:

    data = payload.get(
        "data",
        {},
    )

    records = data.get(
        "indexCloseOnlineRecords",
        [],
    )

    if not records:

        return pd.DataFrame(
            columns=EXPECTED_COLUMNS
        )

    rows = []

    for record in records:

        raw_date = (
            record.get(
                "EOD_TIMESTAMP"
            )
            or record.get(
                "TIMESTAMP"
            )
            or record.get(
                "DATE"
            )
        )

        parsed_date = pd.to_datetime(
            raw_date,
            errors="coerce",
            dayfirst=True,
        )

        if pd.isna(parsed_date):
            continue

        rows.append({
            "date": parsed_date.date(),
            "symbol": "NIFTY50",

            "open": record.get(
                "EOD_OPEN_INDEX_VAL"
            ),

            "high": record.get(
                "EOD_HIGH_INDEX_VAL"
            ),

            "low": record.get(
                "EOD_LOW_INDEX_VAL"
            ),

            "close": record.get(
                "EOD_CLOSE_INDEX_VAL"
            ),

            "volume": None,
        })

    if not rows:

        return pd.DataFrame(
            columns=EXPECTED_COLUMNS
        )

    df = pd.DataFrame(rows)

    return normalize_columns(df)


print(
    nifty50_api_url(
        START_DATE,
        min(
            START_DATE + timedelta(days=89),
            END_DATE,
        ),
    )
)


In [ ]:
# ============================================================
# 15. NIFTY 50 VALIDATION
# ============================================================

def validate_nifty50_dataframe(
    df: pd.DataFrame,
    expected_day: date,
) -> tuple[bool, str]:

    if df.empty:
        return False, "empty_dataframe"

    missing = [
        c
        for c in EXPECTED_COLUMNS
        if c not in df.columns
    ]

    if missing:
        return False, f"missing_columns:{missing}"

    if not (
        df["symbol"] == "NIFTY50"
    ).all():
        return False, "unexpected_symbol"

    if not (
        df["date"] == expected_day
    ).all():
        return False, "wrong_date"

    price_columns = [
        "open",
        "high",
        "low",
        "close",
    ]

    if df[
        price_columns
    ].isna().any().any():

        return False, "missing_ohlc"

    if len(df) != 1:
        return False, (
            f"expected_one_row_got_{len(df)}"
        )

    return True, "ok"


def validate_nifty50_parquet(
    path: Path,
    expected_day: date,
) -> tuple[bool, str, int]:

    if not path.exists():
        return False, "missing_file", 0

    try:

        table = pq.read_table(path)

        df = table.to_pandas()

        df = normalize_columns(df)

        ok, reason = (
            validate_nifty50_dataframe(
                df,
                expected_day,
            )
        )

        return ok, reason, len(df)

    except Exception as exc:

        return (
            False,
            f"parquet_read_error:{type(exc).__name__}:{exc}",
            0,
        )


In [ ]:
# ============================================================
# 16. NIFTY 50 SYNC
# ============================================================

def sync_nifty50(
    start_date: date,
    end_date: date,
    chunk_days: int = 90,
):

    initialize_nse_session()

    current = start_date
    results = []

    while current <= end_date:

        chunk_end = min(
            current + timedelta(
                days=chunk_days - 1
            ),
            end_date,
        )

        logger.info(
            "Fetching NIFTY 50: %s -> %s",
            current,
            chunk_end,
        )

        missing_dates = []

        for day in trading_day_candidates(
            current,
            chunk_end,
        ):

            path = (
                nifty50_parquet_path_for_day(
                    day
                )
            )

            if not path.exists():

                missing_dates.append(day)
                continue

            ok, reason, rows = (
                validate_nifty50_parquet(
                    path,
                    day,
                )
            )

            if ok:
                continue

            if REPAIR_INVALID_FILES:

                logger.warning(
                    "Invalid NIFTY50 file %s: %s",
                    path,
                    reason,
                )

                path.unlink(
                    missing_ok=True
                )

                missing_dates.append(day)

            else:

                results.append({
                    "date": day.isoformat(),
                    "status": "invalid_skipped",
                    "rows": rows,
                    "path": str(path),
                    "message": reason,
                })

        if not missing_dates:

            current = (
                chunk_end
                + timedelta(days=1)
            )

            continue

        url = nifty50_api_url(
            current,
            chunk_end,
        )

        try:

            response = session.get(
                url,
                timeout=REQUEST_TIMEOUT,
            )

            if response.status_code in (
                401,
                403,
            ):

                logger.warning(
                    "NSE returned HTTP %s. "
                    "Refreshing session.",
                    response.status_code,
                )

                initialize_nse_session()

                response = session.get(
                    url,
                    timeout=REQUEST_TIMEOUT,
                )

            response.raise_for_status()

            payload = response.json()

            df = normalize_nifty50_response(
                payload
            )

            if df.empty:

                for day in missing_dates:

                    results.append({
                        "date": day.isoformat(),
                        "status": "not_found",
                        "rows": 0,
                        "path": str(
                            nifty50_parquet_path_for_day(
                                day
                            )
                        ),
                        "message": (
                            "No NIFTY50 data returned; "
                            "likely holiday/non-trading day."
                        ),
                    })

            else:

                for day in missing_dates:

                    day_df = df[
                        df["date"] == day
                    ].copy()

                    path = (
                        nifty50_parquet_path_for_day(
                            day
                        )
                    )

                    if day_df.empty:

                        results.append({
                            "date": day.isoformat(),
                            "status": "not_found",
                            "rows": 0,
                            "path": str(path),
                            "message": (
                                "No NIFTY50 observation "
                                "for this date."
                            ),
                        })

                        continue

                    ok, reason = (
                        validate_nifty50_dataframe(
                            day_df,
                            day,
                        )
                    )

                    if not ok:

                        results.append({
                            "date": day.isoformat(),
                            "status": "error",
                            "rows": 0,
                            "path": str(path),
                            "message": reason,
                        })

                        continue

                    rows = atomic_write_parquet(
                        day_df,
                        path,
                    )

                    results.append({
                        "date": day.isoformat(),
                        "status": "downloaded",
                        "rows": rows,
                        "path": str(path),
                        "message": (
                            "NIFTY50 downloaded successfully."
                        ),
                    })

        except Exception as exc:

            logger.error(
                "NIFTY50 download failed "
                "for %s -> %s: %s",
                current,
                chunk_end,
                exc,
            )

            for day in missing_dates:

                results.append({
                    "date": day.isoformat(),
                    "status": "error",
                    "rows": 0,
                    "path": str(
                        nifty50_parquet_path_for_day(
                            day
                        )
                    ),
                    "message": (
                        f"{type(exc).__name__}: {exc}"
                    ),
                })

        time.sleep(
            SLEEP_BETWEEN_REQUESTS
        )

        current = (
            chunk_end
            + timedelta(days=1)
        )

    return pd.DataFrame(results)


In [ ]:
# ============================================================
# 17. NIFTY 50 MANIFEST
# ============================================================

def save_nifty50_manifest(
    df: pd.DataFrame,
):

    if df.empty:
        return

    manifest = df.copy()

    manifest["checked_at"] = (
        datetime.now().isoformat(
            timespec="seconds"
        )
    )

    if NIFTY50_MANIFEST_FILE.exists():

        old = pd.read_csv(
            NIFTY50_MANIFEST_FILE
        )

        manifest = pd.concat(
            [old, manifest],
            ignore_index=True,
        )

    manifest["date"] = (
        manifest["date"].astype(str)
    )

    manifest = (
        manifest
        .drop_duplicates(
            subset=["date"],
            keep="last",
        )
        .sort_values("date")
    )

    manifest.to_csv(
        NIFTY50_MANIFEST_FILE,
        index=False,
    )


In [ ]:
# ============================================================
# 18. SYNC NIFTY 50
# ============================================================

nifty50_started = datetime.now()

nifty50_results = sync_nifty50(
    START_DATE,
    END_DATE,
    chunk_days=90,
)

nifty50_finished = datetime.now()

save_nifty50_manifest(
    nifty50_results
)

print()
print("=" * 70)
print("NIFTY 50 SYNC COMPLETE")
print("=" * 70)

print("Started :", nifty50_started)
print("Finished:", nifty50_finished)
print("Elapsed :", nifty50_finished - nifty50_started)

if not nifty50_results.empty:

    print("Status:")

    display(
        nifty50_results["status"]
        .value_counts()
        .rename_axis("status")
        .reset_index(
            name="count"
        )
    )

    downloaded_rows = nifty50_results.loc[
        nifty50_results["status"] == "downloaded",
        "rows",
    ].sum()

    print(
        "Downloaded rows:",
        f"{downloaded_rows:,}",
    )

    errors = nifty50_results[
        nifty50_results["status"] == "error"
    ]

    if not errors.empty:

        print("NIFTY50 errors:")

        display(
            errors[
                [
                    "date",
                    "message",
                    "path",
                ]
            ]
        )

print("NIFTY50 directory:", NIFTY50_DIR)

print("NIFTY50 manifest:",NIFTY50_MANIFEST_FILE)


In [ ]:
# ============================================================
# 18. SYNC NIFTY 50
# ============================================================

nifty50_started = datetime.now()

nifty50_results = sync_nifty50(
    START_DATE,
    END_DATE,
    chunk_days=90,
)

nifty50_finished = datetime.now()

save_nifty50_manifest(
    nifty50_results
)

print()
print("=" * 70)
print("NIFTY 50 SYNC COMPLETE")
print("=" * 70)

print("Started :", nifty50_started)
print("Finished:", nifty50_finished)
print("Elapsed :", nifty50_finished - nifty50_started)

if not nifty50_results.empty:

    print("Status:")

    display(
        nifty50_results["status"]
        .value_counts()
        .rename_axis("status")
        .reset_index(
            name="count"
        )
    )

    downloaded_rows = nifty50_results.loc[
        nifty50_results["status"] == "downloaded",
        "rows",
    ].sum()

    print(
        "Downloaded rows:",
        f"{downloaded_rows:,}",
    )

    errors = nifty50_results[
        nifty50_results["status"] == "error"
    ]

    if not errors.empty:

        print("NIFTY50 errors:")

        display(
            errors[
                [
                    "date",
                    "message",
                    "path",
                ]
            ]
        )

print("NIFTY50 directory:", NIFTY50_DIR)

print("NIFTY50 manifest:",NIFTY50_MANIFEST_FILE)


In [ ]:
# ============================================================
# 20. FINAL DATASET INVENTORY
# ============================================================

def parquet_inventory(directory: Path):

    files = sorted(
        directory.glob(
            "year=*/**/*.parquet"
        )
    )

    rows = []

    for path in files:

        try:

            metadata = (
                pq.ParquetFile(
                    path
                ).metadata
            )

            rows.append({
                "file": str(path),
                "rows": metadata.num_rows,
                "size_mb": (
                    path.stat().st_size
                    / (1024 ** 2)
                ),
            })

        except Exception as exc:

            rows.append({
                "file": str(path),
                "rows": None,
                "size_mb": None,
                "error": str(exc),
            })

    return pd.DataFrame(rows)


stock_inventory = parquet_inventory(
    PARQUET_DIR
)

nifty_inventory = parquet_inventory(
    NIFTY50_DIR
)

print("STOCK DATA")
print("-" * 50)
print(
    "Files:",
    len(stock_inventory),
)
print(
    "Rows:",
    f"{stock_inventory['rows'].sum():,.0f}"
    if not stock_inventory.empty
    else 0,
)
print(
    "Size MB:",
    f"{stock_inventory['size_mb'].sum():,.1f}"
    if not stock_inventory.empty
    else 0,
)

print()
print("NIFTY 50 DATA")
print("-" * 50)
print(
    "Files:",
    len(nifty_inventory),
)
print(
    "Rows:",
    f"{nifty_inventory['rows'].sum():,.0f}"
    if not nifty_inventory.empty
    else 0,
)
print(
    "Size MB:",
    f"{nifty_inventory['size_mb'].sum():,.1f}"
    if not nifty_inventory.empty
    else 0,
)


In [ ]:
# ============================================================
# 21. DUCKDB SANITY CHECK
# ============================================================

con = duckdb.connect()

stock_glob = str(
    PARQUET_DIR / "**" / "*.parquet"
)

nifty_glob = str(
    NIFTY50_DIR / "**" / "*.parquet"
)

print("STOCK DATA SUMMARY")

stock_summary = con.execute(
    f'''
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT symbol) AS symbols,
        MIN(date) AS min_date,
        MAX(date) AS max_date
    FROM read_parquet(
        '{stock_glob}',
        hive_partitioning=true
    )
    '''
).df()

display(stock_summary)

print("NIFTY 50 SUMMARY")

nifty_summary = con.execute(
    f'''
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT symbol) AS symbols,
        MIN(date) AS min_date,
        MAX(date) AS max_date,
        MIN(close) AS min_close,
        MAX(close) AS max_close
    FROM read_parquet(
        '{nifty_glob}',
        hive_partitioning=true
    )
    '''
).df()

display(nifty_summary)

print("Latest NIFTY 50 rows:")

latest_nifty = con.execute(
    f'''
    SELECT
        date,
        symbol,
        open,
        high,
        low,
        close,
        volume
    FROM read_parquet(
        '{nifty_glob}',
        hive_partitioning=true
    )
    ORDER BY date DESC
    LIMIT 10
    '''
).df()

display(latest_nifty)


# 22. How to Run

### First test

For a quick test, use:

```python
START_DATE = date(2024, 7, 8)
END_DATE = date(2024, 7, 12)
```

Run all cells and inspect both stock and NIFTY summaries.

### Full historical sync

Then use:

```python
START_DATE = date(2011, 1, 1)
END_DATE = None
```

The notebook is idempotent:

- Valid stock Parquet → skipped
- Missing stock Parquet → downloaded
- Invalid stock Parquet → repaired
- Valid NIFTY 50 Parquet → skipped
- Missing NIFTY 50 Parquet → downloaded
- Invalid NIFTY 50 Parquet → repaired
- NSE 404 / no index record → recorded as non-trading/missing
- Writes are atomic
- Manifests are updated

### Daily use

You can simply rerun the notebook every day with:

```python
START_DATE = date(2011, 1, 1)
END_DATE = None
```

It will only fetch dates that are missing or invalid.

### Important

The NIFTY 50 files are intentionally kept separate from stock files:

```text
data/parquet/
    -> individual NSE securities

data/indices/nifty50/
    -> NIFTY 50 index
```

Both datasets share the same OHLC schema, making them easy to join in the backtesting framework.
